In [ ]:
from typing import List
from pydantic import TypeAdapter, BaseModel
from typing import Dict
import json
from dotenv import load_dotenv


load_dotenv() 

class Page(BaseModel):
    id: str
    name: str
    access_token: str

class UserAccount(BaseModel):
    id: str
    name: str
    access_token: str
    pages: List[Page]


In [29]:

file_path = '//home//maniram//workspace//data//fb.json'
raw_data = "{}"

try:
    with open(file_path, 'r') as file:
        raw_data = json.load(file)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [30]:

# Define the adapter for a Dict with Page objects as values
adapter = TypeAdapter(Dict[str, UserAccount])

# Convert the raw dictionary into a dictionary of Page objects
pages_dict = adapter.validate_python(raw_data)

In [31]:

import time

current_unix_time_int = int(time.time())
print(current_unix_time_int)
SCHEDULED_TIME=current_unix_time_int+(48*3600)
PAGE_ID=pages_dict['mw'].pages[0].id
ACCESS_TOKEN=pages_dict['mw'].pages[0].access_token

1768988089


## Upload photo

In [34]:
import requests

# Define the endpoint and access token
url = f"https://graph.facebook.com/v24.0/{PAGE_ID}/photos"

# Form data (text fields)
data = {
    'published': 'false',
    'temporary': 'true',
    'access_token': ACCESS_TOKEN
}

# File data (image)
# Ensure the path below is correct for your environment
file_path = "img1.png"

try:
    with open(file_path, 'rb') as f:
        files = {
            'source': f
        }
        
        # Make the POST request
        # Note: Do NOT manually set Content-Type to 'multipart/form-data'. 
        # The requests library does this automatically with the correct boundary.
        response = requests.post(url, data=data, files=files)

    # Check the response
    if response.status_code == 200:
        print("Success!")
        print(response.json())
    else:
        print(f"Failed with status code: {response.status_code}")
        print(response.text)

except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


Success!
{'id': '122170117958781957'}


## Schedule post

In [35]:
post_id = response.json()['id']

In [36]:


# API Endpoint for the specific Facebook Page Feed
url = f"https://graph.facebook.com/{PAGE_ID}/feed"
photo_id =  response.json()['id']
response_schedule = None
# Data payload formatted as url-encoded body fields
payload = {
    "message": (
        "Don't let yesterday take up too much of today.\n.\n.\n.\n.\n"
        "#dailymotivation #motivationalquotes #Motivation #Success #Inspiration "
        "#Mindset #DreamBig #StayFocused #KeepGoing #BelieveInYourself "
        "#HardWorkPaysOff #NeverGiveUp #PositiveVibes #GoalGetter "
        "#SelfDiscipline #GrowthMindset #YouGotThis"
    ),
    "attached_media[0]":  json.dumps({"media_fbid": photo_id}),
    "access_token": ACCESS_TOKEN,
    "published": "false",
    "scheduled_publish_time": SCHEDULED_TIME,
    "unpublished_content_type": "SCHEDULED"
}

# The 'data' parameter in requests.post automatically sends the request 
# as 'application/x-www-form-urlencoded'
response_schedule = requests.post(url, data=payload)

# Output the result
if response.status_code == 200:
    print("Post scheduled successfully!")
    print("Response:", response_schedule.json())
else:
    print(f"Failed to schedule post. Status code: {response.status_code}")
    print("Error details:", response.text)
response_schedule.json()

Post scheduled successfully!
Response: {'id': '528528303685416_122170117988781957', 'post_supports_client_mutation_id': True}


{'id': '528528303685416_122170117988781957',
 'post_supports_client_mutation_id': True}